# Ms. Pac-Man DQN Agent — Class 3 Assignment

This notebook trains a Deep Q-Network (DQN) to play Atari **Ms. Pac-Man**, following the assignment spec:

- Chooses **exploration rate**, **episode count**, and **learning rate** as the three required hyperparameters (see the *Hyperparameters* cell below — clearly marked).
- Records an **untrained** and a **trained** gameplay GIF.
- Plots **score, loss, and exploration** across training.
- Reports **5 baseline** and **5 trained** evaluation scores (same 5 seeds, same 5% exploration, same time limit, both times) with means.
- Saves **intermediate GIFs and checkpoints** every 1,000 episodes (training runs well past the 25-episode threshold that triggers this requirement).

**On top of plain DQN**, this notebook adds five lightweight, well-established upgrades — Double DQN, a Dueling network architecture, reward scaling instead of hard clipping, Prioritized Experience Replay, and N-step returns — all validated on Ms. Pac-Man specifically in the original research, and all added because they improve *how well and how efficiently* the agent learns, not just *how long* it trains. See the markdown cells above the network and replay buffer definitions for why.

**Revision note (v2):** the first version of this notebook (250 episodes, no PER/N-step) reached a mean trained score of 854 — a solid 7.6x improvement over baseline, but well short of the ~3000 reference score, which turns out to come from the published Double DQN paper after **50 million** training steps (roughly 300x more than that first run used). Since 250 episodes only took ~3 minutes on a T4 GPU, this version spends the same free GPU much more fully — `EPISODES` is now `6000` — and adds PER + N-step to squeeze more learning out of each episode. See the *Hyperparameters* cell for the full reasoning.

**How to run this:** Runtime → Change runtime type → select a GPU (T4 is fine) → Runtime → Run all. Expect roughly 1–2 hours end to end — the training cell prints a running time estimate every 100 episodes so you can track progress. Every code cell has a plain-language comment explaining what it does.


In [1]:
# --- Install packages not already in the Colab image ---
# (torch and opencv usually ship with Colab already; this is safe to re-run either way)
!pip install -q "gymnasium[atari]" ale-py stable-baselines3 imageio opencv-python-headless


In [ ]:
# --- Imports & device setup ---
import os
import random
import time
from collections import deque

import ale_py
import gymnasium as gym
import imageio
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from stable_baselines3.common.atari_wrappers import AtariWrapper

gym.register_envs(ale_py)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if DEVICE.type != "cuda":
    print("WARNING: no GPU detected. Go to Runtime > Change runtime type > GPU, then Runtime > Restart and run all.")

# Where we'll save everything this notebook produces, so it can be zipped up
# and committed to the GitHub repo afterward.
OUTPUT_DIR = "outputs"
GIF_DIR = os.path.join(OUTPUT_DIR, "gifs")
CKPT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
for d in (GIF_DIR, CKPT_DIR, PLOT_DIR):
    os.makedirs(d, exist_ok=True)


## Hyperparameters

These are the **three hyperparameters the assignment requires us to choose**, plus a short justification for each — updated from v1 now that the goal is closing the gap to the professor's ~3000 reference score, not just demonstrating the pipeline works:

- **`LEARNING_RATE`** — how big a step the network takes each time it updates its beliefs. Unchanged at `0.0001`, the assignment's own suggested reference value and the standard value from the original DQN research. This wasn't the bottleneck in v1 (loss was still decreasing steadily), so it's left alone.
- **`EXPLORATION`** — the *post-warmup* random-action frequency. Lowered from `0.1` to `0.05`: with roughly 24x more training episodes than v1, the agent gets far more chances to refine a good policy, so it's worth leaning a bit more toward exploiting what it's learned rather than continuing to explore at the same rate as a much shorter run. This was Silvia's own instinct after seeing the v1 result — real, but a minor lever on its own (a few percent, not the main driver of the score gap).
- **`EPISODES`** — how many full training games to play. Raised from `250` to `6000`. This is the main lever: v1's 250 episodes took only ~3 minutes on a T4 GPU, meaning the original 1-2 hour time budget was barely used. A classmate's public repo for this same assignment reported a mean trained score of 1360 using 2000 episodes — so 6000 is a deliberate, evidence-based push past that, while still fitting the original time budget.

Everything else below is a **supporting setting**, not one of the three graded hyperparameters. In v1 these were scaled ~10x down from "textbook" DQN defaults because that run was short; now that the step budget is far larger, several are scaled back up — but still nowhere near full research-scale (matching the literature's ~3000 score would need roughly 50 million training steps, about 300x this run's budget, which isn't feasible in a single Colab session).


In [ ]:
# --- The three required hyperparameters ---
LEARNING_RATE = 0.0001   # reference value suggested by the assignment; unchanged from v1
EXPLORATION = 0.05       # post-warmup random-action frequency; lowered from v1's 0.1 (see markdown above)
EPISODES = 6000          # training episodes; raised from v1's 250 (see markdown above)

# --- Supporting settings (rescaled for a ~24x bigger run than v1) ---
EPSILON_START = 1.0
EPSILON_DECAY_EPISODES = 1500    # anneal from EPSILON_START down to EXPLORATION over the first 25% of training
GAMMA = 0.99                     # how much future reward matters vs. immediate reward
BATCH_SIZE = 32
REPLAY_BUFFER_CAPACITY = 100_000 # up from v1's 20,000 - still well under full-scale DQN's ~1,000,000
WARMUP_STEPS = 5_000             # up from v1's 2,000
TARGET_UPDATE_EVERY_STEPS = 2_000  # up from v1's 750, for more stable targets over a longer run
N_STACK = 4                      # how many recent frames the network sees at once (lets it perceive motion)
REWARD_SCALE = 100.0             # divide raw game points by this instead of clipping to -1/0/+1 (see note below)
CHECKPOINT_EVERY_EPISODES = 1000 # intermediate GIF + checkpoint cadence (assignment requires this once episodes > 25)
PROGRESS_EVERY_EPISODES = 100    # how often to print a training progress line

# --- Prioritized Experience Replay (new in v2): replay "surprising" transitions more often ---
N_STEP = 3            # bootstrap 3 steps ahead instead of 1 (see markdown above the replay buffer)
PER_ALPHA = 0.6        # how strongly priority affects sampling (0 = uniform random, like v1)
PER_BETA_START = 0.4  # importance-sampling correction strength at the start of training...
PER_BETA_END = 1.0     # ...annealed up to full correction by the end of training
PER_EPS = 1e-5         # tiny constant so a transition with zero error is never sampled with probability zero

# --- Evaluation settings: MUST stay identical between baseline and post-training eval ---
EVAL_SEEDS = [0, 1, 2, 3, 4]
EVAL_EXPLORATION = 0.05          # per assignment spec: keep a small residual exploration during eval
EVAL_MAX_STEPS = 10_000          # hard cap per eval episode so a stuck run can't hang the notebook

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)


## Environment setup

Two environment configurations, built from the same wrappers:

- **Training environment**: ends an episode as soon as Pac-Man loses a life (`terminal_on_life_loss=True`). This is a standard DQN training trick — it gives the network more frequent, clearer learning signal (dying is an unambiguous bad outcome) rather than waiting through all 3 lives before it gets feedback.
- **Evaluation environment**: plays the **full game, all 3 lives** (`terminal_on_life_loss=False`). This matters because the assignment's "score" — and the leaderboard/professor's ~300-point reference — means the total score across a whole game, not just one life. Using the training-style environment for evaluation would silently under-report the real score.

Both use the same underlying preprocessing (grayscale, resized to 84x84, 4 stacked frames) via Stable-Baselines3's proven `AtariWrapper`, reused here just for its preprocessing — not its training loop, so our own training loop below stays fully visible and editable.

Reward scaling: standard DQN clips every reward to exactly -1, 0, or +1, which would make eating a 10-point dot look identical to eating a 1600-point ghost. We instead divide raw rewards by `REWARD_SCALE` — this keeps the *relative* size of rewards intact (important in Ms. Pac-Man specifically) while still keeping the numbers small enough for stable learning.


In [ ]:
def make_env(training: bool, render_mode=None):
    env = gym.make("ALE/MsPacman-v5", render_mode=render_mode)
    env = AtariWrapper(env, clip_reward=False, terminal_on_life_loss=training)
    env = gym.wrappers.FrameStackObservation(env, stack_size=N_STACK)
    return env


def preprocess(obs):
    # (4, 84, 84, 1) uint8 -> (4, 84, 84) float32 scaled to [0, 1]
    return np.asarray(obs, dtype=np.float32).squeeze(-1) / 255.0


## The network: Dueling architecture + Double DQN

**Dueling architecture**: instead of one output per action, the network splits into a *value* stream (how good is this situation, period) and an *advantage* stream (how much better is each specific move than average, right now). They're recombined into the final action values. This learns faster in states where most moves are equally fine and only one or two really matter — a very common situation in Ms. Pac-Man (e.g., only the move away from a nearby ghost matters; the others are all "fine").

**Double DQN** (used in the training loop below, not here): normal DQN tends to be overly optimistic about how good its own choices are, because the same network both *picks* the best next move and *judges* how good that move is — errors compound. Double DQN fixes this by having the *online* network pick the move and the separate, slower-moving *target* network judge it.


In [ ]:
class DuelingQNetwork(nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(N_STACK, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU(),
        )
        with torch.no_grad():
            conv_out_size = self.conv(torch.zeros(1, N_STACK, 84, 84)).flatten(1).shape[1]

        self.value_head = nn.Sequential(nn.Linear(conv_out_size, 512), nn.ReLU(), nn.Linear(512, 1))
        self.advantage_head = nn.Sequential(nn.Linear(conv_out_size, 512), nn.ReLU(), nn.Linear(512, n_actions))

    def forward(self, x):
        features = self.conv(x).flatten(1)
        value = self.value_head(features)
        advantage = self.advantage_head(features)
        # Recombine: subtracting the mean advantage keeps value/advantage identifiable
        # (otherwise the split is ambiguous - infinitely many value/advantage pairs give the same Q).
        return value + (advantage - advantage.mean(dim=1, keepdim=True))


## Replay buffer: Prioritized Experience Replay + N-step returns

**Prioritized Experience Replay (PER)**: v1 sampled past experience uniformly at random. PER instead replays the transitions the network was most *surprised* by more often — measured by how wrong its prediction was (the TD error) — so training time is spent where there's the most to learn from, rather than re-visiting moments it already predicts well. This is implemented with a *sum tree*: a small binary-tree data structure that lets us both sample proportionally to priority and update a priority, in `O(log n)` time, which matters once the buffer holds 100,000 transitions.

Two details that make PER correct rather than just "resample the hard stuff": (1) new transitions are added with the *highest* priority seen so far, so everything gets tried at least once; (2) because sampling is no longer uniform, each sampled transition's contribution to the loss is corrected by an *importance-sampling weight* (`PER_BETA`, annealed from partial to full correction over training) — otherwise the network would learn a biased picture of the world.

**N-step returns**: instead of learning from one step at a time ("this move, then bootstrap from the very next state"), we accumulate `N_STEP` (3) real steps of reward before bootstrapping. Real observed reward is more informative than the network's own guess, so a few real steps before bootstrapping typically speeds up learning — especially early in training when the network's guesses are still poor.


In [ ]:
class SumTree:
    """Binary tree where each leaf is a transition's priority and each internal
    node is the sum of its children - lets us sample proportionally to priority
    and update a priority, both in O(log n) instead of O(n)."""

    def __init__(self, capacity):
        self.capacity = capacity
        self.tree = np.zeros(2 * capacity - 1)
        self.data = [None] * capacity
        self.write = 0
        self.n_entries = 0

    def _propagate(self, idx, change):
        parent = (idx - 1) // 2
        self.tree[parent] += change
        if parent != 0:
            self._propagate(parent, change)

    def update(self, idx, priority):
        change = priority - self.tree[idx]
        self.tree[idx] = priority
        self._propagate(idx, change)

    def add(self, priority, data):
        idx = self.write + self.capacity - 1
        self.data[self.write] = data
        self.update(idx, priority)
        self.write = (self.write + 1) % self.capacity
        self.n_entries = min(self.n_entries + 1, self.capacity)

    def total(self):
        return self.tree[0]

    def get(self, cumsum):
        idx = 0
        while True:
            left = 2 * idx + 1
            right = left + 1
            if left >= len(self.tree):
                break
            if cumsum <= self.tree[left]:
                idx = left
            else:
                cumsum -= self.tree[left]
                idx = right
        data_idx = idx - self.capacity + 1
        return idx, self.tree[idx], self.data[data_idx]


class PrioritizedReplayBuffer:
    def __init__(self, capacity, alpha):
        self.tree = SumTree(capacity)
        self.alpha = alpha
        self.max_priority = 1.0

    def push(self, state, action, reward, next_state, done):
        # New transitions get the highest priority seen so far, so everything gets tried at least once.
        self.tree.add(self.max_priority ** self.alpha, (state, action, reward, next_state, done))

    def sample(self, batch_size, beta):
        batch, idxs, priorities = [], [], []
        segment = self.tree.total() / batch_size
        for i in range(batch_size):
            cumsum = random.uniform(segment * i, segment * (i + 1))
            idx, priority, data = self.tree.get(cumsum)
            batch.append(data)
            idxs.append(idx)
            priorities.append(priority)

        sampling_probs = np.array(priorities) / self.tree.total()
        weights = (self.tree.n_entries * sampling_probs) ** (-beta)
        weights /= weights.max()  # normalize so the largest weight is 1 (only scales the loss down, never up)

        states, actions, rewards, next_states, dones = zip(*batch)
        batch_tensors = (
            torch.tensor(np.array(states), dtype=torch.float32, device=DEVICE),
            torch.tensor(actions, dtype=torch.int64, device=DEVICE),
            torch.tensor(rewards, dtype=torch.float32, device=DEVICE),
            torch.tensor(np.array(next_states), dtype=torch.float32, device=DEVICE),
            torch.tensor(dones, dtype=torch.float32, device=DEVICE),
        )
        weights_t = torch.tensor(weights, dtype=torch.float32, device=DEVICE)
        return batch_tensors, idxs, weights_t

    def update_priorities(self, idxs, td_errors):
        for idx, td_error in zip(idxs, td_errors):
            priority = (abs(td_error) + PER_EPS) ** self.alpha
            self.max_priority = max(self.max_priority, priority)
            self.tree.update(idx, priority)

    def __len__(self):
        return self.tree.n_entries


class NStepAccumulator:
    """Holds the last N_STEP raw transitions so we can emit an N-step return
    (real observed reward for N_STEP steps, then bootstrap) instead of a 1-step one."""

    def __init__(self, n_step, gamma):
        self.n_step = n_step
        self.gamma = gamma
        self.window = deque(maxlen=n_step)

    def append(self, state, action, reward, next_state, done):
        self.window.append((state, action, reward, next_state, done))

    def ready(self):
        return len(self.window) == self.n_step or (len(self.window) > 0 and self.window[-1][4])

    def pop_transition(self):
        state, action, _, _, _ = self.window[0]
        n_step_return = 0.0
        for i, (_, _, reward, _, done) in enumerate(self.window):
            n_step_return += (self.gamma ** i) * reward
            if done:
                break
        _, _, _, next_state, done = self.window[-1]
        self.window.popleft()
        return state, action, n_step_return, next_state, done

    def flush(self):
        while self.window:
            yield self.pop_transition()


def epsilon_greedy_action(q_net, state, epsilon, n_actions):
    if random.random() < epsilon:
        return random.randrange(n_actions)
    with torch.no_grad():
        state_t = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        return int(torch.argmax(q_net(state_t), dim=1).item())


def double_dqn_loss(online_net, target_net, batch, weights, gamma, n_step):
    states, actions, rewards, next_states, dones = batch
    q_values = online_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
    with torch.no_grad():
        # Double DQN: online net picks the action, target net judges its value.
        next_actions = online_net(next_states).argmax(dim=1)
        next_q = target_net(next_states).gather(1, next_actions.unsqueeze(1)).squeeze(1)
        # gamma**n_step because `rewards` already sums n_step real steps of discounted reward.
        target = rewards + (gamma ** n_step) * next_q * (1.0 - dones)

    td_errors = (target - q_values).detach()
    loss = (weights * F.smooth_l1_loss(q_values, target, reduction="none")).mean()
    return loss, td_errors.cpu().numpy()


## Evaluation function

One function, used identically for the baseline (untrained) and post-training evaluation, so the comparison is fair. It always uses the **same 5 seeds**, the **same 5% exploration**, and the **same step cap** — per the assignment's explicit requirement to keep these unchanged before and after training. It can optionally record one of the runs as a GIF.


In [ ]:
def evaluate_agent(q_net, seeds, epsilon, max_steps, record_seed=None, gif_path=None):
    """Runs one full (all-lives) game per seed and returns the list of scores.
    If record_seed is given, also saves a GIF of that one seed's game."""
    scores = []
    for seed in seeds:
        render_mode = "rgb_array" if (record_seed is not None and seed == record_seed) else None
        env = make_env(training=False, render_mode=render_mode)
        obs, info = env.reset(seed=seed)
        state = preprocess(obs)
        frames = [env.render()] if render_mode else None

        total_reward = 0.0
        done = False
        steps = 0
        while not done and steps < max_steps:
            action = epsilon_greedy_action(q_net, state, epsilon, env.action_space.n)
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            state = preprocess(obs)
            total_reward += reward
            steps += 1
            if frames is not None:
                frames.append(env.render())

        if frames is not None and gif_path is not None:
            imageio.mimsave(gif_path, frames, fps=30)

        scores.append(total_reward)
        env.close()
    return scores


## Baseline evaluation (untrained network)

Before any training, we evaluate the freshly-initialized (essentially random) network on the 5 fixed seeds, and save a GIF of one of those games. This is our "before" measurement.


In [ ]:
sample_env = make_env(training=False)
N_ACTIONS = sample_env.action_space.n
sample_env.close()

online_net = DuelingQNetwork(N_ACTIONS).to(DEVICE)
target_net = DuelingQNetwork(N_ACTIONS).to(DEVICE)
target_net.load_state_dict(online_net.state_dict())
optimizer = torch.optim.Adam(online_net.parameters(), lr=LEARNING_RATE)
replay_buffer = PrioritizedReplayBuffer(REPLAY_BUFFER_CAPACITY, PER_ALPHA)

baseline_scores = evaluate_agent(
    online_net, EVAL_SEEDS, EVAL_EXPLORATION, EVAL_MAX_STEPS,
    record_seed=EVAL_SEEDS[0], gif_path=os.path.join(GIF_DIR, "baseline_untrained.gif"),
)
print("Baseline (untrained) scores:", baseline_scores)
print("Baseline mean score:", np.mean(baseline_scores))


## Training

DQN training loop with Double DQN's target computation, the Dueling network, Prioritized Experience Replay, and N-step returns all defined above. Progress prints every `PROGRESS_EVERY_EPISODES` (100) episodes, including a running time estimate for the full run so you can tell early on whether `EPISODES` needs adjusting.

Every `CHECKPOINT_EVERY_EPISODES` (1000) episodes, we save the model weights and a gameplay GIF — this is the assignment's "intermediate GIFs and checkpoints" requirement for runs past 25 episodes. **If you need to stop training early** (e.g. it's taking longer than expected), the *Resume from a checkpoint* cell right after this one explains how to pick up from the latest saved checkpoint and still get a complete submission.


In [ ]:
episode_scores = []   # training-env score per episode (ends on first life lost - see note below)
episode_losses = []
episode_epsilons = []

total_steps = 0
start_time = time.time()

for episode in range(EPISODES):
    # Linear anneal from EPSILON_START down to EXPLORATION, then hold steady.
    decay_progress = min(1.0, episode / max(1, EPSILON_DECAY_EPISODES))
    epsilon = EPSILON_START + decay_progress * (EXPLORATION - EPSILON_START)
    # Anneal PER's importance-sampling correction from partial to full over training.
    beta = PER_BETA_START + (PER_BETA_END - PER_BETA_START) * (episode / EPISODES)

    env = make_env(training=True)
    obs, info = env.reset(seed=1000 + episode)  # distinct from eval seeds (0-4)
    state = preprocess(obs)
    nstep_acc = NStepAccumulator(N_STEP, GAMMA)

    ep_reward = 0.0
    ep_losses = []
    done = False
    while not done:
        action = epsilon_greedy_action(online_net, state, epsilon, N_ACTIONS)
        next_obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        next_state = preprocess(next_obs)

        nstep_acc.append(state, action, reward / REWARD_SCALE, next_state, float(done))
        if nstep_acc.ready():
            replay_buffer.push(*nstep_acc.pop_transition())
        state = next_state
        ep_reward += reward
        total_steps += 1

        if len(replay_buffer) >= max(WARMUP_STEPS, BATCH_SIZE):
            batch, idxs, weights = replay_buffer.sample(BATCH_SIZE, beta)
            loss, td_errors = double_dqn_loss(online_net, target_net, batch, weights, GAMMA, N_STEP)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            replay_buffer.update_priorities(idxs, td_errors)
            ep_losses.append(loss.item())

        if total_steps % TARGET_UPDATE_EVERY_STEPS == 0:
            target_net.load_state_dict(online_net.state_dict())

    # Flush any transitions still buffered for N-step returns at episode end (shorter horizon near the end).
    for transition in nstep_acc.flush():
        replay_buffer.push(*transition)

    env.close()

    episode_scores.append(ep_reward)
    episode_losses.append(float(np.mean(ep_losses)) if ep_losses else float("nan"))
    episode_epsilons.append(epsilon)

    if (episode + 1) % PROGRESS_EVERY_EPISODES == 0:
        elapsed = time.time() - start_time
        eta = elapsed / (episode + 1) * (EPISODES - episode - 1)
        print(f"episode {episode + 1}/{EPISODES} | score={ep_reward:.0f} | "
              f"mean_loss={episode_losses[-1]:.5f} | epsilon={epsilon:.3f} | "
              f"elapsed={elapsed/60:.1f}min | eta={eta/60:.1f}min")

    if (episode + 1) % CHECKPOINT_EVERY_EPISODES == 0:
        ckpt_path = os.path.join(CKPT_DIR, f"checkpoint_ep{episode + 1}.pt")
        torch.save(online_net.state_dict(), ckpt_path)
        gif_path = os.path.join(GIF_DIR, f"intermediate_ep{episode + 1}.gif")
        evaluate_agent(online_net, [EVAL_SEEDS[0]], EVAL_EXPLORATION, EVAL_MAX_STEPS,
                        record_seed=EVAL_SEEDS[0], gif_path=gif_path)
        print(f"  -> saved checkpoint and GIF at episode {episode + 1}")

print(f"Training complete. Total wall-clock time: {(time.time() - start_time)/60:.1f} minutes, "
      f"{total_steps} environment steps, "
      f"{sum(1 for l in episode_losses if not np.isnan(l))} episodes with learning updates.")


### Resume from a checkpoint (safe to always run, including under Run all)

This cell only does something if training was interrupted early (e.g. Runtime → Interrupt execution, or a disconnect) — it checks whether the training loop above actually finished all `EPISODES`, and only then loads the most recent checkpoint. If training completed normally, this cell prints a message and changes nothing, so it's safe to leave in place under Run all rather than needing to be manually skipped.


In [ ]:
if len(episode_scores) >= EPISODES:
    print(f"Training completed all {EPISODES} episodes normally - nothing to resume, online_net unchanged.")
else:
    checkpoints = sorted(
        (f for f in os.listdir(CKPT_DIR) if f.startswith("checkpoint_ep")),
        key=lambda f: int(f.removeprefix("checkpoint_ep").removesuffix(".pt")),
    )
    if checkpoints:
        latest = checkpoints[-1]
        online_net.load_state_dict(torch.load(os.path.join(CKPT_DIR, latest), map_location=DEVICE))
        print(f"Training only reached {len(episode_scores)}/{EPISODES} episodes. Loaded {latest} - "
              "update the README to note training was interrupted at this point.")
    else:
        print(f"Training only reached {len(episode_scores)}/{EPISODES} episodes, and no checkpoint "
              f"was saved yet (first one saves at episode {CHECKPOINT_EVERY_EPISODES}). "
              "online_net still reflects whatever partial training happened.")


**Note on `episode_scores` above**: because the *training* environment ends each episode after the first life is lost (see *Environment setup*), these per-episode scores are naturally lower than a full 3-life game score, and are used here only to visualize the training trend — not as the graded comparison. The graded baseline-vs-trained comparison below always uses the full-game evaluation environment.


## Post-training evaluation

Exact same 5 seeds, exploration rate, and step cap as the baseline evaluation above — nothing about the evaluation setup changes, only the network being evaluated.


In [ ]:
trained_scores = evaluate_agent(
    online_net, EVAL_SEEDS, EVAL_EXPLORATION, EVAL_MAX_STEPS,
    record_seed=EVAL_SEEDS[0], gif_path=os.path.join(GIF_DIR, "trained.gif"),
)
print("Trained scores:", trained_scores)
print("Trained mean score:", np.mean(trained_scores))

torch.save(online_net.state_dict(), os.path.join(CKPT_DIR, "final_model.pt"))


## Results summary


In [ ]:
print(f"{'Seed':<6}{'Baseline':<12}{'Trained':<12}")
for seed, b, t in zip(EVAL_SEEDS, baseline_scores, trained_scores):
    print(f"{seed:<6}{b:<12.1f}{t:<12.1f}")
print("-" * 30)
print(f"{'Mean':<6}{np.mean(baseline_scores):<12.1f}{np.mean(trained_scores):<12.1f}")
print(f"\nImprovement over baseline: {np.mean(trained_scores) - np.mean(baseline_scores):.1f} points "
      f"({(np.mean(trained_scores) / max(np.mean(baseline_scores), 1e-8) - 1) * 100:.1f}%)")


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 10), sharex=True)

axes[0].plot(episode_scores)
axes[0].set_ylabel("Training score\n(per life, see note above)")
axes[0].set_title("Training progress")

# Raw per-episode loss is noisy; a rolling average is far more readable.
losses = np.array(episode_losses, dtype=float)
window = 10
if len(losses) >= window:
    rolling = np.convolve(np.nan_to_num(losses), np.ones(window) / window, mode="valid")
    axes[1].plot(range(window - 1, len(losses)), rolling)
else:
    axes[1].plot(losses)
axes[1].set_ylabel(f"Loss\n({window}-episode rolling avg)")

axes[2].plot(episode_epsilons)
axes[2].set_ylabel("Exploration rate (epsilon)")
axes[2].set_xlabel("Episode")

plt.tight_layout()
plot_path = os.path.join(PLOT_DIR, "training_curves.png")
plt.savefig(plot_path, dpi=150)
plt.show()
print("Saved plot to", plot_path)


## Next steps: getting these results into the GitHub repo

1. **File → Download → Download .ipynb** — save this executed notebook (with all outputs visible) into the local repo folder, replacing the empty starter notebook.
2. Run the cell below to zip up the `outputs/` folder (GIFs, checkpoints, plot), then download that zip from the Colab file browser (folder icon on the left).
3. Send both files back — I'll unzip the outputs into the repo, fill in the actual numbers in `README.md`, and push everything to GitHub.


In [ ]:
import shutil
shutil.make_archive("outputs", "zip", OUTPUT_DIR)
print("Created outputs.zip - download it from the Colab file browser (folder icon, left sidebar).")
